In [1]:
import pyreadstat
import pandas as pd
import numpy as np

Читаем изначальные данные:

In [2]:

df, meta = pyreadstat.read_sav("data/r33iall_84.sav")
df


,idind,ccredid_i,ccid_i,ccid_h,bbid_i,bbid_h,aaid_i,aaid_h,zid_i,zid_h,...,ccl43,ccl43a,ccl43as,ccm96,ccm97,ccm98,ccm99,ccm100,ccm101,ccm111
0,92.0,487801.0,1008401.0,10084.0,1008401.0,10084.0,1008401.0,10084.0,1008401.0,10084.0,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,108.0,487901.0,1008801.0,10088.0,1008801.0,10088.0,1008801.0,10088.0,1008801.0,10088.0,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,110.0,485703.0,1009603.0,10096.0,1009603.0,10096.0,1009603.0,10096.0,1009603.0,10096.0,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,112.0,488001.0,1008901.0,10089.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,125.0,130301.0,1010801.0,10108.0,1010801.0,10108.0,1010801.0,10108.0,1010801.0,10108.0,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16877,65365.0,285811.0,77002411.0,770024.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16878,65366.0,284504.0,77000404.0,770004.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16879,65367.0,285006.0,77000906.0,770009.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16880,65368.0,285007.0,77000907.0,770009.0,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,NaN,,2.0,1.0,2.0,2.0,2.0,2.0,NaN


Определяем нужные нам переменные:

outcome_vars - переменные исхода (ответы респондентов на вопросы о наличии хронических заболеваний и общая самооценка здоровья - self_assessment_health);

educ - переменная воздействия (образования);

control_vars - контрольные переменные

region - регион

In [3]:
dfw = df.copy()

outcome_vars = ["ccm20.61", "ccm20.62", "ccm20.63", "ccm20.64", "ccm20.65", "ccm20.66", "ccm20.620", "ccm20.69",
                "ccm20.610", "ccm20.611", "ccm20.612", "ccm20.613", "ccm20.615", "ccm20.616", "ccm20.617", "ccm20.618",
                "ccm20.614", "ccm20.619", "ccm20.67"]

self_assessment_health = ["ccm3"]

educ = ["cc_diplom"]

control_vars = ["cc_age", "ccj13.2", "ccj72.172", "cch5", "status", "ccm20.7", "cc_marst", "ccl5.0", "ccj1", "ccm80.0", "ccm71", "ccm113a"]

region = ["region"]

Получаем полный список нужных нам переменных:

In [4]:
full_pre_list = outcome_vars + self_assessment_health + educ + control_vars + region
full_pre_list

['ccm20.61',
 'ccm20.62',
 'ccm20.63',
 'ccm20.64',
 'ccm20.65',
 'ccm20.66',
 'ccm20.620',
 'ccm20.69',
 'ccm20.610',
 'ccm20.611',
 'ccm20.612',
 'ccm20.613',
 'ccm20.615',
 'ccm20.616',
 'ccm20.617',
 'ccm20.618',
 'ccm20.614',
 'ccm20.619',
 'ccm20.67',
 'ccm3',
 'cc_diplom',
 'cc_age',
 'ccj13.2',
 'ccj72.172',
 'cch5',
 'status',
 'ccm20.7',
 'cc_marst',
 'ccl5.0',
 'ccj1',
 'ccm80.0',
 'ccm71',
 'ccm113a',
 'region']

Функции для преобразований значений переменных в нужный нам формат:

In [5]:
def transform_value(x):
    if x == 1:
        return 1
    elif x == 2:
        return 0
    else:
        return np.nan
    
def transform_health(x):
    if x == 1 or x == 2:
        return 1
    elif x == 3 or x == 4 or x == 5:
        return 0
    else:
        return np.nan
    
def transform_health_sharp(x):
    if x == 1:
        return 1
    elif x == 2 or x == 3 or x == 4 or x == 5:
        return 0
    else:
        return np.nan
    

def transform_educ(x):
    if x == 6:
        return 1
    elif x == 1 or x == 2 or x == 3 or x == 4 or x == 5:
        return 0
    else:
        return np.nan
    

def transform_cont(x):
    if x == 99999997 or x == 99999998 or x == 99999999:
        return np.nan
    else:
        return x
    

def transform_area(x):
    if x == 1 or x == 2:
        return 1
    else:
        return 0
    
def transform_inv(x):
    if x == 1 or x == 5 or x == 99999996:
        return 1
    elif x == 2:
        return 0
    else:
        return np.nan
    
def transform_mar(x):
    if x == 2 or x == 3:
        return 1
    elif x == 1 or x == 4 or x == 5 or x == 6:
        return 0
    else:
        return np.nan
    
def transform_doctor(x):
    if x == 1 or x == 2 or x == 3:
        return 1
    elif x == 4 or x == 5:
        return 0
    else:
        return np.nan
    
def transform_work(x):
    if x == 1  or x == 2 or x == 3 or x == 4:
        return 1
    elif x == 5:
        return 0
    else:
        return np.nan




Преобразование части переменных:

In [6]:
dfw = dfw[full_pre_list]

for var in outcome_vars:
    dfw[var] = dfw[var].apply(transform_value)

dfw["is_health_good"] = dfw["ccm3"].apply(transform_health)
dfw["is_health_very_good"] = dfw["ccm3"].apply(transform_health_sharp)
dfw = dfw.drop(columns=self_assessment_health)
dfw["diploma"] = dfw["cc_diplom"].apply(transform_educ)
dfw = dfw.drop(columns=educ)
dfw

,ccm20.61,ccm20.62,ccm20.63,ccm20.64,ccm20.65,ccm20.66,ccm20.620,ccm20.69,ccm20.610,ccm20.611,...,cc_marst,ccl5.0,ccj1,ccm80.0,ccm71,ccm113a,region,is_health_good,is_health_very_good,diploma
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,2.0,5.0,5.0,2.0,2.0,2.0,1.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,...,5.0,3.0,5.0,1.0,2.0,2.0,1.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.0,4.0,1.0,1.0,1.0,2.0,1.0,0.0,0.0,0.0
3,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,...,5.0,1.0,5.0,2.0,2.0,2.0,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,2.0,4.0,1.0,1.0,2.0,2.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16877,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,2.0,3.0,5.0,2.0,2.0,2.0,77.0,0.0,0.0,1.0
16878,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,2.0,5.0,1.0,2.0,2.0,2.0,77.0,0.0,0.0,0.0
16879,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,5.0,1.0,2.0,2.0,2.0,77.0,1.0,0.0,1.0
16880,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,NaN,3.0,NaN,NaN,NaN,NaN,77.0,1.0,0.0,NaN


Словарь для переименования переменных в более осмысленные названия:

In [7]:
rename_dict = {
 'ccm20.61': 'heart',
 'ccm20.62': 'lungs',
 'ccm20.63': 'liver',
 'ccm20.64': 'kidneys',
 'ccm20.65': 'stomach',
 'ccm20.66': 'spine',
 'ccm20.620': 'diabetes',
 'ccm20.69': 'hypertension',
 'ccm20.610': 'joints',
 'ccm20.611': 'ENT_organs',
 'ccm20.612': 'neurology',
 'ccm20.613': 'eyes',
 'ccm20.615': 'allergy',
 'ccm20.616': 'veins',
 'ccm20.617': 'skin',
 'ccm20.618': 'oncology',
 'ccm20.614': 'gynecology',
 'ccm20.619': 'urine-reproductive_system',
 'ccm20.67': 'others',
 'cc_age': 'age',
 'ccj13.2': 'income',
 'ccj72.172': 'n_child',
 'cch5': 'sex',
 'status': 'type_area',
 'ccm20.7': 'invalid',
 'cc_marst': 'mar_st',
 'ccl5.0': 'visit_doctor',
 'ccj1': 'work',
 'ccm80.0': 'alcohol',
 'ccm71': 'smoking',
 'ccm113a': 'phys_active'
}

Переименование + преобразование:

In [8]:
dfw = dfw.rename(columns=rename_dict)

dfw["age"] = dfw["age"].apply(transform_cont)
dfw["income"] = dfw["income"].apply(transform_cont)
dfw["n_child"] = dfw["n_child"].apply(transform_cont)
dfw["type_area"] = dfw["type_area"].apply(transform_area)
dfw["invalid"] = dfw["invalid"].apply(transform_inv)
dfw["mar_st"] = dfw["mar_st"].apply(transform_mar)
dfw["visit_doctor"] = dfw["visit_doctor"].apply(transform_doctor)
dfw["work"] = dfw["work"].apply(transform_work)
dfw["alcohol"] = dfw["alcohol"].apply(transform_value)
dfw["smoking"] = dfw["smoking"].apply(transform_value)
dfw["phys_active"] = dfw["phys_active"].apply(transform_value)


Все переменные, кроме переменных исхода:

In [9]:
main = ['age', 'income', 'n_child', 'sex', 'type_area', 'invalid',
       'mar_st', 'visit_doctor', 'work', 'alcohol', 'smoking', 'phys_active',
       'is_health_good', 'is_health_very_good', 'diploma']

Пробуем удалить все наблюдения, у которых есть хоть одно nan-значение:

In [10]:
dfw.dropna()

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,mar_st,visit_doctor,work,alcohol,smoking,phys_active,region,is_health_good,is_health_very_good,diploma
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0
13,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,9.0,0.0,0.0,1.0
14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,1.0,0.0,0.0,1.0,9.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16846,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,142.0,0.0,0.0,0.0
16855,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,142.0,0.0,0.0,0.0
16861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,142.0,0.0,0.0,0.0
16878,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0


Осталось меньше 3 000 строк, что кажется недостаточным. Смотрим, какие переменные проблемные:

In [11]:
dfw.isna().sum()

heart                           86
lungs                           80
liver                           98
kidneys                        104
stomach                        113
spine                          108
diabetes                        96
hypertension                    84
joints                          96
ENT_organs                      77
neurology                      100
eyes                            85
allergy                         83
veins                           89
skin                            80
oncology                       111
gynecology                    7388
urine-reproductive_system      369
others                         413
age                              0
income                       10456
n_child                       6709
sex                              0
type_area                        0
invalid                         12
mar_st                        2743
visit_doctor                   243
work                          2738
alcohol             

Видим, что много nan-значений в gynecology, urine-reproductive_system и others. Эти переменные не является существенными (к тому же часть из них относятся лишь к одному полу). Их исключение не повлияет сильно на предмет исследования, но зато поможет убрать источники nan-значений.

In [12]:
df_clear = dfw.drop(columns=["gynecology", "urine-reproductive_system", "others"]).dropna()
df_clear_with_region = df_clear.copy()
df_clear = df_clear_with_region.drop(columns=["region"])
display(df_clear)
display(df_clear_with_region)

,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,invalid,mar_st,visit_doctor,work,alcohol,smoking,phys_active,is_health_good,is_health_very_good,diploma
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16855,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
16861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
16868,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
16878,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


,heart,lungs,liver,kidneys,stomach,spine,diabetes,hypertension,joints,ENT_organs,...,mar_st,visit_doctor,work,alcohol,smoking,phys_active,region,is_health_good,is_health_very_good,diploma
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16855,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,1.0,0.0,1.0,142.0,0.0,0.0,0.0
16861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,1.0,0.0,0.0,142.0,0.0,0.0,0.0
16868,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0
16878,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0


Теперь после  удаления трех переменных и всех строк с nan-значениями у нас 4598 наблюдений. Вполне нормально. Сохраняем:

In [13]:
df_clear.to_csv('data/final_dataset.csv', index=False)

In [14]:
df_clear_with_region.to_csv('probit/probit_dataset.csv', index=False)